# Step5 Eigenspace 对比：Mat32 / eps / top_q

目标：
- 复用 `v1_mat32_eps_topq_sanity.ipynb` 的数据与参数
- 在步骤 5（估计矩阵 A 的 top eigenspace）比较不同方法的耗时与残差
- 基于不同 eigenspace 再跑完整算法（预条件 + PCG），检查准确性与稳定性

说明：
- 结果只打印在 notebook 中，不写 CSV
- 计算量很大，请确认内存与 CPU 资源

Result:
Surrogate Method, replacing original A with a coarser/looser/subsample can significantly increase the speed of estimation of eigenspace without loosing precision. And current estimation method is good enough.

## Prepare

In [28]:
import os
import sys
import time
import traceback
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

# Ensure project root is importable no matter where notebook kernel starts.
_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.discretization import GridSpec
from efgp_eigenpro_py.benchmark import (
    make_dataset,
    make_test_set,
    true_func_2d,
    compute_rmse,
)
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext, ensure_gpu_data_context
from efgp_eigenpro_py.gpu.iterative_solvers import pcg_solve_gpu
from efgp_eigenpro_py.gpu.v1_ops import (
    apply_A_v1,
    predict_v1,
    solve_beta_plain_cg_v1,
)
from efgp_eigenpro_py.gpu.v2_preconditioner import GPUPreconditionerData, apply_preconditioner_v2
from efgp_eigenpro_py.gpu.surrogate_ops import (
    EigenspaceConfig,
    estimate_top_eigenspace_v3,
    gpu_precompute_v1,
)
from efgp_eigenpro_py.gpu.versions import GPURunConfig

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: d:\NU\ML\efgp_eigenpro_py\gpu\sanity_check
sys.path[0]: D:\NU\ML


In [29]:
# ----- user requested test matrix -----
REG_LAMBDA = 0.1
N_TRAIN = 1_000_00
DIM = 2
LENGTHSCALE = 0.1
NU = 1.5  # Mat32
EPS_LIST = [1e-6]
TOP_Q_LIST = [64]

N_TEST = 20_000
NOISE = 0.02
SEED_TRAIN = 20230
SEED_TEST = 2027

SOLVE_TOL = 1e-6
GPU_TOL = SOLVE_TOL
GPU_MAXITER = 3000
GPU_CHUNK_SIZE = None
GPU_DEBUG_FINITE_CHECKS = False
GPU_NUFFT = "auto"
L2_SCALED = True
OVERSAMPLE_DEFAULT = 16

BASELINE_EIG_METHOD = "subspace_iter"
ENABLE_CUSTOM_EIGENSPACE = True
CUSTOM_METHODS_ENABLED = ["custom_rand_lanczos"]
# ["custom_block_lanczos", "custom_rand_lanczos", "custom_lobpcg_precond"]
CUSTOM_BLOCK_POWER_CFG = {
    "tol": 1e-6,
    "maxiter": 8,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
}
CUSTOM_RAND_SUBSPACE_CFG = {
    "tol": 1e-6,
    "maxiter": 1,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
}
CUSTOM_BLOCK_LANCZOS_CFG = {
    "krylov_depth": 2,
    "restart_cycles": 2,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
}
CUSTOM_RAND_LANCZOS_CFG = {
    "krylov_depth": 2,
    "restart_cycles": 2,
    "rand_iters": 1,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
}
CUSTOM_LOBPCG_PRECOND_CFG = {
    "tol": 1e-6,
    "maxiter": 40,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "precond": None,
}
COMBO_GRID_VALUES = {
    "grid_scale": [0.001,0.002,0.005,0.01,0.02,0.03,0.04,0.1], #0.1
    "eps_factor": [ 100.0],
    "subsample_frac": [0.3],
}
COMBO_COMMON_CFG = {
    "subsample_seed": 0,
    "sur_iter": 1,
    "refine_iter": 1,
    "oversample": OVERSAMPLE_DEFAULT,
    "block_size": None,
}
COMBO_ENABLED = True  # True/False 或指定 combo 名称列表

SURROGATE_ENABLED = None
# "coarse_grid", "loose_eps", "subsample"
SURROGATE_METHODS = {
    "coarse_grid": {
        "grid_scale": 0.35,
        "grid_m": None,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
    "loose_eps": {
        "eps_factor": 50.0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
    "subsample": {
        "subsample_frac": 0.05,
        "subsample_seed": 0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
}


def _combo_tag(val: float) -> str:
    return str(val).replace(".", "p")


COMBO_VARIANTS = []
for grid_scale in COMBO_GRID_VALUES["grid_scale"]:
    for eps_factor in COMBO_GRID_VALUES["eps_factor"]:
        for subsample_frac in COMBO_GRID_VALUES["subsample_frac"]:
            combo_name = (
                f"combo_g{_combo_tag(grid_scale)}"
                f"_e{_combo_tag(eps_factor)}"
                f"_s{_combo_tag(subsample_frac)}"
            )
            combo_cfg = {
                "name": combo_name,
                "kind": "combo",
                "grid_scale": grid_scale,
                "grid_m": None,
                "eps_factor": eps_factor,
                "subsample_frac": subsample_frac,
                **COMBO_COMMON_CFG,
            }
            COMBO_VARIANTS.append(combo_cfg)

EIG_METHODS = [
    {
        "name": "subspace_iter",    # actually use, v3_subspace_iter
        "method": "v3_subspace_iter",
        "n_iter": 3,
        "block_size": None,
        "oversample": OVERSAMPLE_DEFAULT,
        "callable": None,
   }
]

if SURROGATE_ENABLED is not None:
    for sur_name in SURROGATE_ENABLED:
        sur_cfg = SURROGATE_METHODS.get(sur_name)
        if sur_cfg is None:
            continue
        EIG_METHODS.append(
            {
                "name": f"sur_{sur_name}",
                "method": "surrogate",
                "surrogate": {"name": sur_name, **sur_cfg},
                "callable": None,
            }
        )

if COMBO_ENABLED:
    allowed = None
    if isinstance(COMBO_ENABLED, (list, tuple, set)):
        allowed = {name for name in COMBO_ENABLED}
    for combo_cfg in COMBO_VARIANTS:
        if allowed is not None and combo_cfg["name"] not in allowed:
            continue
        EIG_METHODS.append(
            {
                "name": f"sur_{combo_cfg['name']}",
                "method": "surrogate",
                "surrogate": combo_cfg,
                "callable": None,
            }
        )

'''
    {
        "name": "eigsh",
        "method": "eigsh",
        "tol": 1e-6,
        "maxiter": 200,
        "block_size": None,
        "oversample": OVERSAMPLE_DEFAULT,
        "callable": None,
    },
    {
        "name": "lobpcg",
        "method": "lobpcg",
        "tol": 1e-6,
        "maxiter": 40,
        "block_size": None,
        "oversample": OVERSAMPLE_DEFAULT,
        "callable": None,
    },
'''
RUN_FULL_SOLVE = True

print("Run mode: print-only (no CSV output)")
print("EPS_LIST:", EPS_LIST)
print("TOP_Q_LIST:", TOP_Q_LIST)
print("EIG_METHODS:", [cfg["name"] for cfg in EIG_METHODS])
print("ENABLE_CUSTOM_EIGENSPACE:", ENABLE_CUSTOM_EIGENSPACE)
print("CUSTOM_METHODS_ENABLED:", CUSTOM_METHODS_ENABLED)
print("CUSTOM_BLOCK_POWER_CFG:", CUSTOM_BLOCK_POWER_CFG)
print("CUSTOM_RAND_SUBSPACE_CFG:", CUSTOM_RAND_SUBSPACE_CFG)
print("CUSTOM_BLOCK_LANCZOS_CFG:", CUSTOM_BLOCK_LANCZOS_CFG)
print("CUSTOM_RAND_LANCZOS_CFG:", CUSTOM_RAND_LANCZOS_CFG)
print("CUSTOM_LOBPCG_PRECOND_CFG:", CUSTOM_LOBPCG_PRECOND_CFG)
print("SURROGATE_ENABLED:", SURROGATE_ENABLED)
print("COMBO_ENABLED:", COMBO_ENABLED)
print("COMBO_GRID_VALUES:", COMBO_GRID_VALUES)
print("COMBO_COMMON_CFG:", COMBO_COMMON_CFG)
print("COMBO_VARIANTS:", len(COMBO_VARIANTS))
print("SURROGATE_METHODS:", SURROGATE_METHODS)
print("BASELINE_EIG_METHOD:", BASELINE_EIG_METHOD)
print("GPU_NUFFT:", GPU_NUFFT)
print("RUN_FULL_SOLVE:", RUN_FULL_SOLVE)

Run mode: print-only (no CSV output)
EPS_LIST: [1e-06]
TOP_Q_LIST: [64]
EIG_METHODS: ['subspace_iter', 'sur_combo_g0p001_e100p0_s0p3', 'sur_combo_g0p002_e100p0_s0p3', 'sur_combo_g0p005_e100p0_s0p3', 'sur_combo_g0p01_e100p0_s0p3', 'sur_combo_g0p02_e100p0_s0p3', 'sur_combo_g0p03_e100p0_s0p3', 'sur_combo_g0p04_e100p0_s0p3', 'sur_combo_g0p1_e100p0_s0p3']
ENABLE_CUSTOM_EIGENSPACE: True
CUSTOM_METHODS_ENABLED: ['custom_rand_lanczos']
CUSTOM_BLOCK_POWER_CFG: {'tol': 1e-06, 'maxiter': 8, 'block_size': None, 'oversample': 16}
CUSTOM_RAND_SUBSPACE_CFG: {'tol': 1e-06, 'maxiter': 1, 'block_size': None, 'oversample': 16}
CUSTOM_BLOCK_LANCZOS_CFG: {'krylov_depth': 2, 'restart_cycles': 2, 'block_size': None, 'oversample': 16}
CUSTOM_RAND_LANCZOS_CFG: {'krylov_depth': 2, 'restart_cycles': 2, 'rand_iters': 1, 'block_size': None, 'oversample': 16}
CUSTOM_LOBPCG_PRECOND_CFG: {'tol': 1e-06, 'maxiter': 40, 'block_size': None, 'oversample': 16, 'precond': None}
SURROGATE_ENABLED: None
COMBO_ENABLED: True
CO

In [30]:
# 数据准备（请确认机器内存/CPU 资源足够）
print("Building dataset ...")
x_train, y_train = make_dataset(DIM, N_TRAIN, true_func_2d, noise=NOISE, seed=SEED_TRAIN)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)

kernel = make_matern(lengthscale=LENGTHSCALE, nu=NU, dim=DIM, variance=1.0)
print(kernel)

Building dataset ...
x_train: (100000, 2) float64
y_train: (100000,) float64
x_test : (20000, 2) float64
KernelSpec(fam='matern', dim=2, lengthscale=0.1, variance=1.0, nu=1.5)


In [31]:
def _mse_mae(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    diff = a - b
    mse = float(np.mean(np.abs(diff) ** 2))
    mae = float(np.mean(np.abs(diff)))
    return mse, mae


def _device_to_numpy(xp, arr):
    if hasattr(xp, "asnumpy"):
        return np.asarray(xp.asnumpy(arr))
    if hasattr(arr, "get"):
        return np.asarray(arr.get())
    return np.asarray(arr)


def _resolve_block_size(top_q, cfg):
    block_size = cfg.get("block_size")
    if block_size is None:
        oversample = int(cfg.get("oversample", 2))
        block_size = max(top_q + oversample, top_q + 1)
    return int(block_size)


def _apply_matvec_block(matvec, matvec_block, V, xp):
    if matvec_block is not None:
        return matvec_block(V)
    cols = [matvec(V[:, i]) for i in range(V.shape[1])]
    return xp.stack(cols, axis=1)


def _randn(xp, seed, shape):
    try:
        rng = xp.random.RandomState(seed)
        return rng.standard_normal(shape)
    except Exception:
        return xp.random.standard_normal(shape)


def _sort_eigpairs(eigvals, eigvecs, xp):
    idx = xp.argsort(xp.real(eigvals))[::-1]
    return xp.real(eigvals[idx]), eigvecs[:, idx]


# 自定义 eigenspace 方法移到下一单元。

def _normalize_eigpairs(result, xp):
    if isinstance(result, tuple):
        eigvals, eigvecs = result
    else:
        if not hasattr(result, "values") or not hasattr(result, "vectors"):
            raise ValueError("custom eigenspace must return values and vectors.")
        eigvals, eigvecs = result.values, result.vectors
    return SimpleNamespace(values=xp.asarray(eigvals), vectors=xp.asarray(eigvecs))


def _build_gpu_context(solver, x, y, cfg, grid_override=None):
    backend = build_gpu_backend_bundle(cfg.backend)
    data_ctx = ensure_gpu_data_context(backend, x, y, state=None)
    data_ctx.meta["debug_finite_checks"] = bool(cfg.debug_finite_checks)
    op_ctx = GPUOperatorContext()
    t0 = time.perf_counter()
    data_ctx = gpu_precompute_v1(
        backend,
        solver.kernel,
        solver.eps,
        solver.nufft_tol,
        data_ctx,
        op_ctx,
        l2scaled=solver.l2scaled,
        chunk_size=cfg.chunk_size,
        grid_override=grid_override,
    )
    t1 = time.perf_counter()
    return backend, data_ctx, op_ctx, float(t1 - t0)


def _apply_A_block_gpu(backend, data_ctx, op_ctx, reg_lambda, v_block):
    xp = backend.xp
    v_block = xp.asarray(v_block, dtype=xp.complex128)
    if v_block.ndim == 1:
        v_block = v_block.reshape(-1, 1)
    out_block = xp.empty_like(v_block)
    for i in range(v_block.shape[1]):
        apply_A_v1(
            backend,
            data_ctx,
            v_block[:, i],
            float(reg_lambda),
            op_ctx,
            out=out_block[:, i],
        )
    return out_block


def _build_surrogate_grid(fine_m, fine_h, grid_scale=None, grid_m=None):
    if grid_m is not None:
        m_coarse = int(grid_m)
    else:
        if grid_scale is None:
            grid_scale = 1.0
        m_coarse = max(1, int(fine_m * float(grid_scale)))
    xis = np.arange(-m_coarse, m_coarse + 1) * float(fine_h)
    return GridSpec(xis=xis, h=float(fine_h), mtot=xis.size, hm=m_coarse)


def _embed_coarse_to_fine(U_sur, mtot_sur, mtot_fine, dim, xp):
    if int(mtot_sur) == int(mtot_fine):
        return U_sur
    if mtot_sur > mtot_fine:
        raise ValueError("surrogate grid is larger than fine grid")
    start = (int(mtot_fine) - int(mtot_sur)) // 2
    if start < 0:
        raise ValueError("invalid surrogate/fine grid alignment")

    shape_sur = (int(mtot_sur),) * int(dim)
    shape_fine = (int(mtot_fine),) * int(dim)
    k = int(U_sur.shape[1])
    out = xp.zeros((int(mtot_fine) ** int(dim), k), dtype=U_sur.dtype)
    slicer = tuple(slice(start, start + int(mtot_sur)) for _ in range(int(dim)))

    for j in range(k):
        u_sur = U_sur[:, j].reshape(shape_sur)
        u_fine = xp.zeros(shape_fine, dtype=U_sur.dtype)
        u_fine[slicer] = u_sur
        out[:, j] = u_fine.reshape(-1)
    return out


def _build_surrogate_context(solver, x_full, y_full, gpu_cfg, sur_cfg, fine_meta):
    sur_tag = sur_cfg.get("name", "")
    sur_kind = sur_cfg.get("kind", sur_tag)
    x_use = x_full
    y_use = y_full
    grid_override = None
    eps_sur = float(solver.eps)

    if sur_kind in ("subsample", "combo"):
        frac = float(sur_cfg.get("subsample_frac", 1.0))
        seed = int(sur_cfg.get("subsample_seed", 0))
        n = int(x_full.shape[0])
        if frac < 1.0:
            rng = np.random.default_rng(seed)
            n_keep = max(1, int(n * frac))
            idx = rng.choice(n, size=n_keep, replace=False)
            x_use = x_full[idx]
            y_use = y_full[idx]
    if sur_kind in ("loose_eps", "combo"):
        eps_factor = float(sur_cfg.get("eps_factor", 1.0))
        eps_sur = float(solver.eps) * eps_factor
    if sur_kind in ("coarse_grid", "combo"):
        fine_mtot = int(fine_meta.get("mtot", 0))
        fine_m = int((fine_mtot - 1) // 2)
        fine_h = float(fine_meta.get("h", 0.0))
        grid_scale = sur_cfg.get("grid_scale")
        grid_m = sur_cfg.get("grid_m")
        grid_override = _build_surrogate_grid(fine_m, fine_h, grid_scale, grid_m)

    sur_solver = solver
    if eps_sur != float(solver.eps):
        sur_solver = EFGPSolver(
            kernel=solver.kernel,
            reg_lambda=solver.reg_lambda,
            eps=eps_sur,
            nufft_tol=solver.nufft_tol,
            l2scaled=solver.l2scaled,
        )

    backend_s, data_ctx_s, op_ctx_s, t_pre = _build_gpu_context(
        sur_solver,
        x_use,
        y_use,
        gpu_cfg,
        grid_override=grid_override,
    )

    mtot_sur = int(data_ctx_s.meta.get("mtot", 0))
    info = {
        "surrogate_tag": sur_tag,
        "surrogate_eps": float(eps_sur),
        "surrogate_m": int((mtot_sur - 1) // 2) if mtot_sur else np.nan,
        "surrogate_subsample_n": int(x_use.shape[0]),
        "surrogate_grid_override": bool(grid_override is not None),
        "surrogate_precompute": float(t_pre),
    }
    return sur_solver, backend_s, data_ctx_s, op_ctx_s, info


def _run_surrogate_eigenspace(sur_cfg, backend, data_ctx, op_ctx, top_q, solver, gpu_cfg):
    xp = backend.xp
    q_req = int(top_q) + 1

    sur_solver, backend_s, data_ctx_s, op_ctx_s, sur_info = _build_surrogate_context(
        solver,
        x_train,
        y_train,
        gpu_cfg,
        sur_cfg,
        data_ctx.meta,
    )

    def _apply_block_sur(V):
        return _apply_A_block_gpu(backend_s, data_ctx_s, op_ctx_s, REG_LAMBDA, V)

    sur_block = _resolve_block_size(q_req, sur_cfg)
    sur_iter = int(sur_cfg.get("sur_iter", 1))
    sur_iter = max(sur_iter, 1)
    eig_cfg_sur = EigenspaceConfig(q_max=q_req, block_size=sur_block, n_iter=sur_iter)

    t_s0 = time.perf_counter()
    vals_sur, vecs_sur, diag_sur = estimate_top_eigenspace_v3(
        backend=backend_s,
        apply_A_block_gpu=_apply_block_sur,
        size=int(data_ctx_s.rhs_gpu.size),
        cfg=eig_cfg_sur,
    )
    t_s1 = time.perf_counter()

    U_init = vecs_sur
    mtot_sur = int(data_ctx_s.meta.get("mtot", 0))
    mtot_fine = int(data_ctx.meta.get("mtot", 0))
    dim = int(data_ctx.meta.get("dim", 0))
    if mtot_sur and mtot_fine and mtot_sur != mtot_fine:
        U_init = _embed_coarse_to_fine(U_init, mtot_sur, mtot_fine, dim, xp)

    refine_iter = int(sur_cfg.get("refine_iter", 1))
    refine_iter = max(refine_iter, 1)
    fine_block = _resolve_block_size(q_req, sur_cfg)
    eig_cfg_fine = EigenspaceConfig(q_max=q_req, block_size=fine_block, n_iter=refine_iter)

    def _apply_block_fine(V):
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, V)

    t_r0 = time.perf_counter()
    vals_fine, vecs_fine, diag_fine = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_apply_block_fine,
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg_fine,
        init_Q=U_init,
    )
    t_r1 = time.perf_counter()

    diag = dict(diag_fine)
    diag.update(sur_info)
    diag.update(
        {
            "surrogate_block_size": int(sur_block),
            "surrogate_n_iter": int(sur_iter),
            "surrogate_time_eigenspace": float(t_s1 - t_s0),
            "surrogate_refine_iter": int(refine_iter),
            "surrogate_time_refine": float(t_r1 - t_r0),
            "surrogate_diag_residual": float(diag_sur.get("residual_fro", np.nan)),
            "surrogate_diag_residual_rel": float(diag_sur.get("residual_fro_rel", np.nan)),
        }
    )
    return SimpleNamespace(values=vals_fine, vectors=vecs_fine), fine_block, diag


def _run_cupyx_eigenspace(cfg, backend, data_ctx, op_ctx, q_req):
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for eigsh/lobpcg") from exc

    xp = backend.xp
    size = int(data_ctx.rhs_gpu.size)
    dtype = xp.complex128

    def _matvec(v):
        v = xp.asarray(v, dtype=dtype).reshape(-1)
        out = xp.empty_like(v)
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)
        return out

    def _matmat(X):
        X = xp.asarray(X, dtype=dtype)
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, X)

    A = csl.LinearOperator(
        (size, size),
        matvec=_matvec,
        matmat=_matmat,
        dtype=dtype,
    )
    if cfg["method"] == "eigsh":
        eigvals, eigvecs = csl.eigsh(
            A,
            k=q_req,
            which="LM",
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
        )
    else:
        X = _randn(xp, 0, (size, q_req)).astype(dtype)
        X, _ = xp.linalg.qr(X)
        X = xp.ascontiguousarray(X)
        eigvals, eigvecs = csl.lobpcg(
            A,
            X,
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
            largest=True,
        )
    eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def _run_eigenspace(cfg, backend, data_ctx, op_ctx, top_q, solver, gpu_cfg):
    xp = backend.xp
    q_req = int(top_q) + 1

    def _apply_block(V):
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, V)

    def _apply_vec(v):
        out = _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, v)
        return out.reshape(-1)

    eig_diag = None
    if cfg["method"] == "v3_subspace_iter":
        block_size = _resolve_block_size(q_req, cfg)
        eig_cfg = EigenspaceConfig(
            q_max=q_req,
            block_size=block_size,
            n_iter=int(cfg.get("n_iter", 3)),
        )
        eigvals, eigvecs, eig_diag = estimate_top_eigenspace_v3(
            backend=backend,
            apply_A_block_gpu=_apply_block,
            size=int(data_ctx.rhs_gpu.size),
            cfg=eig_cfg,
        )
        eigpairs = SimpleNamespace(values=eigvals, vectors=eigvecs)
    elif cfg["method"] in ("eigsh", "lobpcg"):
        block_size = _resolve_block_size(q_req, cfg)
        eigpairs = _run_cupyx_eigenspace(cfg, backend, data_ctx, op_ctx, q_req)
    elif cfg["method"] == "surrogate":
        sur_cfg = cfg.get("surrogate", {})
        eigpairs, block_size, eig_diag = _run_surrogate_eigenspace(
            sur_cfg,
            backend,
            data_ctx,
            op_ctx,
            top_q,
            solver,
            gpu_cfg,
        )
    else:
        block_size = _resolve_block_size(q_req, cfg)
        fn = cfg.get("callable")
        if fn is None:
            raise ValueError(f"Unknown eigenspace method: {cfg.get('method')}")
        eigpairs = fn(
            matvec=_apply_vec,
            size=int(data_ctx.rhs_gpu.size),
            top_q=top_q,
            matvec_block=_apply_block,
            block_size=block_size,
            oversample=int(cfg.get("oversample", 2)),
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
            xp=xp,
            cfg=cfg,
        )

    eigpairs = _normalize_eigpairs(eigpairs, xp)
    if eigpairs.values.size < q_req:
        raise ValueError("eigenspace must return at least top_q + 1 values")

    if eig_diag is None:
        res_fro, res_rel = _eigenspace_residual_gpu(
            _apply_block,
            eigpairs.values[:q_req],
            eigpairs.vectors[:, :q_req],
            xp,
        )
        eig_diag = {
            "residual_fro": res_fro,
            "residual_fro_rel": res_rel,
        }

    return eigpairs, block_size, eig_diag


def _eigenspace_residual_gpu(apply_block, eigvals, eigvecs, xp):
    AU = apply_block(eigvecs)
    resid = AU - eigvecs * eigvals.reshape(1, -1)
    res_fro = float(xp.linalg.norm(resid))
    au_norm = float(xp.linalg.norm(AU))
    res_rel = res_fro / max(au_norm, 1e-30)
    return res_fro, res_rel


def _build_precond_gpu(backend, eigvals, eigvecs, top_q):
    xp = backend.xp
    eigvals = xp.asarray(eigvals).reshape(-1)
    eigvecs = xp.asarray(eigvecs)
    q = int(top_q)
    if eigvals.size <= q:
        mu = float(eigvals[-1])
    else:
        mu = float(eigvals[q])
    scale = xp.asarray(1.0 - (mu / eigvals[:q]))
    precond = GPUPreconditionerData(
        U_gpu=eigvecs[:, :q],
        UH_gpu=eigvecs[:, :q].conj().T,
        scale_gpu=scale,
        scale_col_gpu=scale.reshape(-1, 1),
    )
    return precond, mu


def _run_pcg_gpu(backend, data_ctx, op_ctx, precond_data):
    def _matvec(v, out):
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)

    def _precond(v, out):
        apply_preconditioner_v2(backend, precond_data, v, op_ctx=op_ctx, out=out)

    return pcg_solve_gpu(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        GPU_TOL,
        GPU_MAXITER,
        return_stats=True,
    )


def _predict_gpu_host(backend, data_ctx, x_eval, beta_gpu):
    yhat_gpu = predict_v1(backend, data_ctx, x_eval, beta_gpu)
    return _device_to_numpy(backend.xp, yhat_gpu)


eigen esti

In [32]:
# 自定义 eigenspace 方法写在这里（可自行扩展）

def custom_block_power(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    k = int(top_q) + 1
    n_iter = 8 if maxiter is None else int(maxiter)
    Q = _randn(xp, 0, (size, k)).astype(xp.complex128)
    for _ in range(n_iter):
        Z = _apply_matvec_block(matvec, matvec_block, Q, xp)
        Q, _ = xp.linalg.qr(Z)
    AQ = _apply_matvec_block(matvec, matvec_block, Q, xp)
    eigvals = xp.sum(xp.conj(Q) * AQ, axis=0)
    eigvals, eigvecs = _sort_eigpairs(eigvals, Q, xp)
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def custom_randomized_subspace(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    k = int(top_q) + 1
    oversample = int(oversample or 0)
    l = k + oversample
    n_iter = 1 if maxiter is None else int(maxiter)
    Q = _randn(xp, 1, (size, l)).astype(xp.complex128)
    for _ in range(max(n_iter, 1)):
        Y = _apply_matvec_block(matvec, matvec_block, Q, xp)
        Q, _ = xp.linalg.qr(Y)
    B = Q.conj().T @ _apply_matvec_block(matvec, matvec_block, Q, xp)
    eigvals, eigvecs_small = xp.linalg.eigh(B)
    eigvals = xp.real(eigvals)
    idx = xp.argsort(eigvals)[::-1]
    eigvals = eigvals[idx][:k]
    eigvecs = Q @ eigvecs_small[:, idx][:, :k]
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def _orthonormalize_block(Q, basis, xp):
    for B in basis:
        Q = Q - B @ (B.conj().T @ Q)
    Q, _ = xp.linalg.qr(Q)
    return Q


def _pad_block(Q, block, xp):
    if Q.shape[1] >= block:
        return Q[:, :block]
    extra = _randn(xp, 2, (Q.shape[0], block - Q.shape[1])).astype(xp.complex128)
    Q = xp.concatenate([Q, extra], axis=1)
    Q, _ = xp.linalg.qr(Q)
    return Q


def _block_krylov(matvec_block, Q0, n_steps, xp):
    basis = []
    Q = Q0
    for _ in range(max(int(n_steps), 1)):
        Q = _orthonormalize_block(Q, basis, xp)
        basis.append(Q)
        Q = matvec_block(Q)
    V = xp.concatenate(basis, axis=1)
    AV = matvec_block(V)
    T = V.conj().T @ AV
    T = 0.5 * (T + T.conj().T)
    eigvals, eigvecs = xp.linalg.eigh(T)
    idx = xp.argsort(eigvals)[::-1]
    eigvals = xp.real(eigvals[idx])
    eigvecs = eigvecs[:, idx]
    U = V @ eigvecs
    return eigvals, U


def _block_lanczos_core(matvec_block, Q0, k, block, krylov_depth, restart_cycles, xp):
    Q = _pad_block(Q0, block, xp)
    eigvals = None
    U = None
    for _ in range(max(int(restart_cycles), 1)):
        eigvals, U = _block_krylov(matvec_block, Q, krylov_depth, xp)
        U = U[:, :k]
        Q = _pad_block(U, block, xp)
    return eigvals[:k], U[:, :k]


def custom_block_lanczos(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    k = int(top_q) + 1
    block = int(block_size) if block_size is not None else int(top_q + oversample)
    krylov_depth = int(cfg.get("krylov_depth", 2)) if cfg else 2
    restart_cycles = int(cfg.get("restart_cycles", 2)) if cfg else 2

    if matvec_block is None:
        def matvec_block(V):
            return _apply_matvec_block(matvec, None, V, xp)

    Q0 = _randn(xp, 3, (size, block)).astype(xp.complex128)
    Q0, _ = xp.linalg.qr(Q0)
    eigvals, eigvecs = _block_lanczos_core(
        matvec_block,
        Q0,
        k,
        block,
        krylov_depth,
        restart_cycles,
        xp,
    )
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def custom_rand_lanczos(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    k = int(top_q) + 1
    block = int(block_size) if block_size is not None else int(top_q + oversample)
    krylov_depth = int(cfg.get("krylov_depth", 2)) if cfg else 2
    restart_cycles = int(cfg.get("restart_cycles", 2)) if cfg else 2
    rand_iters = int(cfg.get("rand_iters", 1)) if cfg else 1

    if matvec_block is None:
        def matvec_block(V):
            return _apply_matvec_block(matvec, None, V, xp)

    Q0 = _randn(xp, 4, (size, block)).astype(xp.complex128)
    for _ in range(max(rand_iters, 1)):
        Q0 = matvec_block(Q0)
        Q0, _ = xp.linalg.qr(Q0)

    eigvals, eigvecs = _block_lanczos_core(
        matvec_block,
        Q0,
        k,
        block,
        krylov_depth,
        restart_cycles,
        xp,
    )
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def custom_lobpcg_precond(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for lobpcg") from exc

    k = int(top_q) + 1
    block = int(block_size) if block_size is not None else int(top_q + oversample)

    def _matvec(v):
        v = xp.asarray(v, dtype=xp.complex128).reshape(-1)
        return matvec(v)

    def _matmat(V):
        return _apply_matvec_block(matvec, matvec_block, V, xp)

    A = csl.LinearOperator((size, size), matvec=_matvec, matmat=_matmat, dtype=xp.complex128)
    X = _randn(xp, 5, (size, block)).astype(xp.complex128)
    X, _ = xp.linalg.qr(X)

    precond = None
    if cfg is not None:
        precond = cfg.get("precond")
    M = None
    if precond is not None:
        def _pvec(v):
            v = xp.asarray(v, dtype=xp.complex128).reshape(-1)
            return precond(v)
        M = csl.LinearOperator((size, size), matvec=_pvec, dtype=xp.complex128)

    eigvals, eigvecs = csl.lobpcg(
        A,
        X,
        M=M,
        tol=float(tol),
        maxiter=maxiter,
        largest=True,
    )
    eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
    return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])


CUSTOM_EIGENSPACE_METHODS = [
    {
        "name": "custom_block_lanczos",
        "method": "custom",
        "block_size": CUSTOM_BLOCK_LANCZOS_CFG["block_size"],
        "oversample": CUSTOM_BLOCK_LANCZOS_CFG["oversample"],
        "krylov_depth": CUSTOM_BLOCK_LANCZOS_CFG["krylov_depth"],
        "restart_cycles": CUSTOM_BLOCK_LANCZOS_CFG["restart_cycles"],
        "callable": custom_block_lanczos,
    },
    {
        "name": "custom_rand_lanczos",
        "method": "custom",
        "block_size": CUSTOM_RAND_LANCZOS_CFG["block_size"],
        "oversample": CUSTOM_RAND_LANCZOS_CFG["oversample"],
        "krylov_depth": CUSTOM_RAND_LANCZOS_CFG["krylov_depth"],
        "restart_cycles": CUSTOM_RAND_LANCZOS_CFG["restart_cycles"],
        "rand_iters": CUSTOM_RAND_LANCZOS_CFG["rand_iters"],
        "callable": custom_rand_lanczos,
    },
    {
        "name": "custom_lobpcg_precond",
        "method": "custom",
        "tol": CUSTOM_LOBPCG_PRECOND_CFG["tol"],
        "maxiter": CUSTOM_LOBPCG_PRECOND_CFG["maxiter"],
        "block_size": CUSTOM_LOBPCG_PRECOND_CFG["block_size"],
        "oversample": CUSTOM_LOBPCG_PRECOND_CFG["oversample"],
        "precond": CUSTOM_LOBPCG_PRECOND_CFG["precond"],
        "callable": custom_lobpcg_precond,
    },
    {
        "name": "custom_block_power",
        "method": "custom",
        "tol": CUSTOM_BLOCK_POWER_CFG["tol"],
        "maxiter": CUSTOM_BLOCK_POWER_CFG["maxiter"],
        "block_size": CUSTOM_BLOCK_POWER_CFG["block_size"],
        "oversample": CUSTOM_BLOCK_POWER_CFG["oversample"],
        "callable": custom_block_power,
    },
    {
        "name": "custom_rand_subspace",
        "method": "custom",
        "tol": CUSTOM_RAND_SUBSPACE_CFG["tol"],
        "maxiter": CUSTOM_RAND_SUBSPACE_CFG["maxiter"],
        "block_size": CUSTOM_RAND_SUBSPACE_CFG["block_size"],
        "oversample": CUSTOM_RAND_SUBSPACE_CFG["oversample"],
        "callable": custom_randomized_subspace,
    },
]


def _register_custom_methods():
    existing = {cfg.get("name") for cfg in EIG_METHODS}
    allowed = CUSTOM_METHODS_ENABLED
    if allowed is not None:
        allowed = {name for name in allowed}
    for cfg in CUSTOM_EIGENSPACE_METHODS:
        if cfg["name"] in existing:
            continue
        if allowed is not None and cfg["name"] not in allowed:
            continue
        EIG_METHODS.append(cfg)


if ENABLE_CUSTOM_EIGENSPACE:
    _register_custom_methods()


## Step 1: 估计 A 的 top eigenspace（算法步骤 5）

- 这里只做 GPU eigenspace 估计与速度/残差对比（NUFFT 仍在 CPU）
- 自定义方法请在 `custom_block_lanczos`/`custom_rand_lanczos`/`custom_lobpcg_precond`/`custom_block_power`/`custom_randomized_subspace` 中实现，并将 `ENABLE_CUSTOM_EIGENSPACE=True`


In [33]:
eig_rows = []
eig_cache = {}
eig_metrics = {}
state_cache = {}

for eps in EPS_LIST:
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    gpu_cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=GPU_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    backend, data_ctx, op_ctx, precompute_time = _build_gpu_context(
        solver,
        x_train,
        y_train,
        gpu_cfg,
    )
    state_cache[eps] = {
        "solver": solver,
        "gpu_cfg": gpu_cfg,
        "backend": backend,
        "data_ctx": data_ctx,
        "op_ctx": op_ctx,
        "time_precompute": precompute_time,
    }

    mtot = int(data_ctx.meta.get("mtot", 0))
    m = int((mtot - 1) // 2) if mtot else np.nan

    for top_q in TOP_Q_LIST:
        top_q = int(top_q)
        if top_q <= 0:
            continue
        for cfg in EIG_METHODS:
            method_name = cfg["name"]
            sur_cfg = cfg.get("surrogate", {}) if cfg.get("method") == "surrogate" else {}
            oversample = int(sur_cfg.get("oversample", cfg.get("oversample", 2)))
            block_size = _resolve_block_size(top_q + 1, sur_cfg or cfg)
            try:
                t_e0 = time.perf_counter()
                eigpairs, block_size, eig_diag = _run_eigenspace(
                    cfg,
                    backend,
                    data_ctx,
                    op_ctx,
                    top_q,
                    solver,
                    gpu_cfg,
                )
                t_e1 = time.perf_counter()

                eigvals_host = _device_to_numpy(backend.xp, eigpairs.values)
                lambda_1 = float(np.real(eigvals_host[0]))
                lambda_q = float(np.real(eigvals_host[top_q - 1]))
                lambda_qp1 = float(
                    np.real(eigvals_host[min(top_q, eigvals_host.size - 1)])
                )

                surrogate_eig_time = float(eig_diag.get("surrogate_time_eigenspace", np.nan))
                surrogate_refine_time = float(eig_diag.get("surrogate_time_refine", np.nan))
                time_eig = float(t_e1 - t_e0)
                if cfg.get("method") == "surrogate":
                    time_eig = surrogate_eig_time + surrogate_refine_time

                row = {
                    "eps": eps,
                    "top_q": top_q,
                    "m": m,
                    "eig_method": method_name,
                    "eig_solver": cfg["method"],
                    "block_size": block_size,
                    "oversample": oversample,
                    "time_precompute": precompute_time,
                    "time_eigenspace": time_eig,
                    "eig_residual_fro": float(eig_diag.get("residual_fro", np.nan)),
                    "eig_residual_fro_rel": float(eig_diag.get("residual_fro_rel", np.nan)),
                    "lambda_1": lambda_1,
                    "lambda_q": lambda_q,
                    "lambda_q_plus_1": lambda_qp1,
                    "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                    "device_name": str(backend.device_name),
                    "surrogate_tag": eig_diag.get("surrogate_tag", ""),
                    "surrogate_eps": float(eig_diag.get("surrogate_eps", np.nan)),
                    "surrogate_m": float(eig_diag.get("surrogate_m", np.nan)),
                    "surrogate_subsample_n": float(eig_diag.get("surrogate_subsample_n", np.nan)),
                    "surrogate_refine_iter": float(eig_diag.get("surrogate_refine_iter", np.nan)),
                    "surrogate_n_iter": float(eig_diag.get("surrogate_n_iter", np.nan)),
                    "surrogate_time_eigenspace": surrogate_eig_time,
                    "surrogate_time_refine": surrogate_refine_time,
                    "surrogate_precompute": float(eig_diag.get("surrogate_precompute", np.nan)),
                    "status": "ok",
                    "error": "",
                }
                eig_rows.append(row)
                eig_cache[(eps, top_q, method_name)] = eigpairs
                eig_metrics[(eps, top_q, method_name)] = row
            except Exception:
                tb = traceback.format_exc()
                print(f"[EIG ERROR] eps={eps}, top_q={top_q}, method={method_name}")
                print(tb)
                eig_rows.append(
                    {
                        "eps": eps,
                        "top_q": top_q,
                        "m": m,
                        "eig_method": method_name,
                        "eig_solver": cfg["method"],
                        "block_size": block_size,
                        "oversample": oversample,
                        "time_precompute": precompute_time,
                        "time_eigenspace": np.nan,
                        "eig_residual_fro": np.nan,
                        "eig_residual_fro_rel": np.nan,
                        "lambda_1": np.nan,
                        "lambda_q": np.nan,
                        "lambda_q_plus_1": np.nan,
                        "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                        "device_name": str(backend.device_name),
                        "surrogate_tag": sur_cfg.get("name", ""),
                        "surrogate_eps": float(sur_cfg.get("eps_factor", np.nan)),
                        "surrogate_m": np.nan,
                        "surrogate_subsample_n": float(sur_cfg.get("subsample_frac", np.nan)),
                        "surrogate_refine_iter": float(sur_cfg.get("refine_iter", np.nan)),
                        "surrogate_n_iter": float(sur_cfg.get("sur_iter", np.nan)),
                        "surrogate_time_eigenspace": np.nan,
                        "surrogate_time_refine": np.nan,
                        "surrogate_precompute": np.nan,
                        "status": "error",
                        "error": tb,
                    }
                )


eig_df = pd.DataFrame(eig_rows)
print("Eigenspace rows:", len(eig_df))
eig_df

[EIG ERROR] eps=1e-06, top_q=64, method=sur_combo_g0p001_e100p0_s0p3
Traceback (most recent call last):
  File "C:\Users\24681\AppData\Local\Temp\ipykernel_35208\831127895.py", line 52, in <module>
    eigpairs, block_size, eig_diag = _run_eigenspace(
                                     ^^^^^^^^^^^^^^^^
  File "C:\Users\24681\AppData\Local\Temp\ipykernel_35208\3846984916.py", line 341, in _run_eigenspace
    eigpairs, block_size, eig_diag = _run_surrogate_eigenspace(
                                     ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\24681\AppData\Local\Temp\ipykernel_35208\3846984916.py", line 212, in _run_surrogate_eigenspace
    vals_sur, vecs_sur, diag_sur = estimate_top_eigenspace_v3(
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\NU\ML\efgp_eigenpro_py\gpu\surrogate_ops.py", line 304, in estimate_top_eigenspace_v3
    raise ValueError("q_max must be < size.")
ValueError: q_max must be < size.

[EIG ERROR] eps=1e-06, top_q=64, method=sur_co

,eps,top_q,m,eig_method,eig_solver,block_size,oversample,time_precompute,time_eigenspace,eig_residual_fro,...,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
0,0.000001,64,181,subspace_iter,v3_subspace_iter,81,16,0.174621,12.915861,103.033257,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,0.000001,64,181,sur_combo_g0p001_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,...,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
2,0.000001,64,181,sur_combo_g0p002_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,...,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
3,0.000001,64,181,sur_combo_g0p005_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,...,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
4,0.000001,64,181,sur_combo_g0p01_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,...,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
5,0.000001,64,181,sur_combo_g0p02_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,...,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
6,0.000001,64,181,sur_combo_g0p03_e100p0_s0p3,surrogate,81,16,0.174621,6.219832,416.942968,...,0.0001,5.0,30000.0,1.0,1.0,0.231696,5.988136,0.010486,ok,
7,0.000001,64,181,sur_combo_g0p04_e100p0_s0p3,surrogate,81,16,0.174621,7.124506,205.573208,...,0.0001,7.0,30000.0,1.0,1.0,0.229012,6.895493,0.013045,ok,
8,0.000001,64,181,sur_combo_g0p1_e100p0_s0p3,surrogate,81,16,0.174621,7.228035,239.987123,...,0.0001,18.0,30000.0,1.0,1.0,0.248154,6.979881,0.011803,ok,
9,0.000001,64,181,custom_rand_lanczos,custom,81,16,0.174621,20.853558,16.426273,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,


In [34]:
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(eig_df.sort_values(["eps", "top_q", "eig_method"]))

,eps,top_q,m,eig_method,eig_solver,block_size,oversample,time_precompute,time_eigenspace,eig_residual_fro,eig_residual_fro_rel,lambda_1,lambda_q,lambda_q_plus_1,nufft_stage,device_name,surrogate_tag,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
9,0.000001,64,181,custom_rand_lanczos,custom,81,16,0.174621,20.853558,16.426273,0.001138,5574.788581,289.756784,250.824246,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
0,0.000001,64,181,subspace_iter,v3_subspace_iter,81,16,0.174621,12.915861,103.033257,0.007141,5574.788576,280.843874,246.015206,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,0.000001,64,181,sur_combo_g0p001_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,NaN,NaN,NaN,NaN,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p001_e100p0_s0p3,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
2,0.000001,64,181,sur_combo_g0p002_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,NaN,NaN,NaN,NaN,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p002_e100p0_s0p3,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
3,0.000001,64,181,sur_combo_g0p005_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,NaN,NaN,NaN,NaN,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p005_e100p0_s0p3,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
4,0.000001,64,181,sur_combo_g0p01_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,NaN,NaN,NaN,NaN,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p01_e100p0_s0p3,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
5,0.000001,64,181,sur_combo_g0p02_e100p0_s0p3,surrogate,81,16,0.174621,NaN,NaN,NaN,NaN,NaN,NaN,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p02_e100p0_s0p3,100.0000,NaN,0.3,1.0,1.0,NaN,NaN,NaN,error,"Traceback (most recent call last):\n File ""C:..."
6,0.000001,64,181,sur_combo_g0p03_e100p0_s0p3,surrogate,81,16,0.174621,6.219832,416.942968,0.028926,5574.788438,222.200712,214.359863,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p03_e100p0_s0p3,0.0001,5.0,30000.0,1.0,1.0,0.231696,5.988136,0.010486,ok,
7,0.000001,64,181,sur_combo_g0p04_e100p0_s0p3,surrogate,81,16,0.174621,7.124506,205.573208,0.014253,5574.788446,229.013204,226.226997,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p04_e100p0_s0p3,0.0001,7.0,30000.0,1.0,1.0,0.229012,6.895493,0.013045,ok,
8,0.000001,64,181,sur_combo_g0p1_e100p0_s0p3,surrogate,81,16,0.174621,7.228035,239.987123,0.016637,5574.782045,255.752784,233.740058,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p1_e100p0_s0p3,0.0001,18.0,30000.0,1.0,1.0,0.248154,6.979881,0.011803,ok,


## Step 2: 完整算法（步骤 6-8）

- 使用相同 GPU eigenspace 构建预条件 + GPU PCG（NUFFT 仍在 CPU）
- 指标：rmse_test、beta_mse、beta_mae、yhat_mse、yhat_mae


In [35]:
full_rows = []
baseline_cache = {}

if RUN_FULL_SOLVE:
    for eps in EPS_LIST:
        cache = state_cache[eps]
        solver = cache["solver"]
        backend = cache["backend"]
        data_ctx = cache["data_ctx"]
        op_ctx = cache["op_ctx"]
        precompute_time = cache["time_precompute"]

        t_cpu0 = time.perf_counter()
        state_cpu = solver.precompute(x_train, y_train)
        t_cpu1 = time.perf_counter()
        _ = float(t_cpu1 - t_cpu0)

        mtot = int(data_ctx.meta.get("mtot", 0))
        m = int((mtot - 1) // 2) if mtot else np.nan

        for top_q in TOP_Q_LIST:
            top_q = int(top_q)
            if top_q <= 0:
                continue
            baseline_key = (eps, top_q)
            baseline_beta = baseline_cache.get(baseline_key)

            for cfg in EIG_METHODS:
                method_name = cfg["name"]
                key = (eps, top_q, method_name)
                eigpairs = eig_cache.get(key)
                if eigpairs is None:
                    continue
                eig_info = eig_metrics.get(key, {})
                try:
                    t_pb0 = time.perf_counter()
                    precond_data, _mu = _build_precond_gpu(
                        backend,
                        eigpairs.values,
                        eigpairs.vectors,
                        top_q,
                    )
                    t_pb1 = time.perf_counter()

                    t_s0 = time.perf_counter()
                    beta_gpu, it, relres, stats = _run_pcg_gpu(
                        backend,
                        data_ctx,
                        op_ctx,
                        precond_data,
                    )
                    t_s1 = time.perf_counter()

                    t_pg0 = time.perf_counter()
                    yhat_gpu_host = _predict_gpu_host(
                        backend,
                        data_ctx,
                        x_test,
                        beta_gpu,
                    )
                    t_pg1 = time.perf_counter()

                    beta_cpu = _device_to_numpy(backend.xp, beta_gpu)
                    t_pc0 = time.perf_counter()
                    yhat_cpu = solver.predict(x_test, beta_cpu, state_cpu)
                    t_pc1 = time.perf_counter()
                    _ = float(t_pc1 - t_pc0)

                    rmse = compute_rmse(yhat_cpu, y_test)
                    yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat_cpu)

                    beta_mse = np.nan
                    beta_mae = np.nan
                    if method_name == BASELINE_EIG_METHOD:
                        baseline_beta = beta_cpu
                        baseline_cache[baseline_key] = baseline_beta
                    elif baseline_beta is not None:
                        beta_mse, beta_mae = _mse_mae(beta_cpu, baseline_beta)

                    time_eigenspace = float(eig_info.get("time_eigenspace", np.nan))
                    time_solve = float(t_s1 - t_s0)
                    n_precond = int(stats.get("n_precond", 0))
                    t_precond_total = float(stats.get("t_precond_total", np.nan))
                    if n_precond > 0 and not np.isnan(t_precond_total):
                        t_precond_avg = float(t_precond_total / max(n_precond, 1))
                    else:
                        t_precond_avg = np.nan
                    if it and int(it) > 0:
                        t_cg_iter_avg = float(time_solve / max(int(it), 1))
                    else:
                        t_cg_iter_avg = np.nan

                    full_rows.append(
                        {
                            "mode": "gpu_precond",
                            "eps": eps,
                            "top_q": top_q,
                            "eig_method": method_name,
                            "m": m,
                            "rmse_test": float(rmse),
                            "beta_mse": float(beta_mse),
                            "beta_mae": float(beta_mae),
                            "yhat_mse": float(yhat_mse),
                            "yhat_mae": float(yhat_mae),
                            "cg_iters": int(it),
                            "cg_relres": float(relres),
                            "n_matvec": int(stats["n_matvec"]),
                            "t_matvec_avg": float(stats["t_matvec_total"] / max(stats["n_matvec"], 1)),
                            "t_matvec_total": float(stats["t_matvec_total"]),
                            "time_precompute": float(precompute_time),
                            "time_eigenspace": time_eigenspace,
                            "time_precond_build": float(t_pb1 - t_pb0),
                            "eig_residual_fro": float(eig_info.get("eig_residual_fro", np.nan)),
                            "eig_residual_fro_rel": float(eig_info.get("eig_residual_fro_rel", np.nan)),
                            "time_solve": time_solve,
                            "time_predict": float(t_pg1 - t_pg0),
                            "wall_s_total": float(
                                precompute_time
                                + time_eigenspace
                                + (t_pb1 - t_pb0)
                                + time_solve
                                + (t_pg1 - t_pg0)
                            ),
                            "n_precond": n_precond,
                            "t_precond_total": t_precond_total,
                            "t_precond_avg": t_precond_avg,
                            "t_cg_iter_avg": t_cg_iter_avg,
                            "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                            "device_name": str(backend.device_name),
                            "surrogate_tag": eig_info.get("surrogate_tag", ""),
                            "surrogate_eps": float(eig_info.get("surrogate_eps", np.nan)),
                            "surrogate_m": float(eig_info.get("surrogate_m", np.nan)),
                            "surrogate_subsample_n": float(eig_info.get("surrogate_subsample_n", np.nan)),
                            "surrogate_refine_iter": float(eig_info.get("surrogate_refine_iter", np.nan)),
                            "surrogate_n_iter": float(eig_info.get("surrogate_n_iter", np.nan)),
                            "surrogate_time_eigenspace": float(eig_info.get("surrogate_time_eigenspace", np.nan)),
                            "surrogate_time_refine": float(eig_info.get("surrogate_time_refine", np.nan)),
                            "surrogate_precompute": float(eig_info.get("surrogate_precompute", np.nan)),
                            "status": "ok",
                            "error": "",
                        }
                    )
                except Exception:
                    tb = traceback.format_exc()
                    print(f"[SOLVE ERROR] eps={eps}, top_q={top_q}, method={method_name}")
                    print(tb)
                    full_rows.append(
                        {
                            "mode": "gpu_precond",
                            "eps": eps,
                            "top_q": top_q,
                            "eig_method": method_name,
                            "m": m,
                            "rmse_test": np.nan,
                            "beta_mse": np.nan,
                            "beta_mae": np.nan,
                            "yhat_mse": np.nan,
                            "yhat_mae": np.nan,
                            "cg_iters": np.nan,
                            "cg_relres": np.nan,
                            "n_matvec": np.nan,
                            "t_matvec_avg": np.nan,
                            "t_matvec_total": np.nan,
                            "time_precompute": float(precompute_time),
                            "time_eigenspace": float(eig_info.get("time_eigenspace", np.nan)),
                            "time_precond_build": np.nan,
                            "eig_residual_fro": float(eig_info.get("eig_residual_fro", np.nan)),
                            "eig_residual_fro_rel": float(eig_info.get("eig_residual_fro_rel", np.nan)),
                            "time_solve": np.nan,
                            "time_predict": np.nan,
                            "wall_s_total": np.nan,
                            "n_precond": np.nan,
                            "t_precond_total": np.nan,
                            "t_precond_avg": np.nan,
                            "t_cg_iter_avg": np.nan,
                            "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                            "device_name": str(backend.device_name),
                            "surrogate_tag": eig_info.get("surrogate_tag", ""),
                            "surrogate_eps": float(eig_info.get("surrogate_eps", np.nan)),
                            "surrogate_m": float(eig_info.get("surrogate_m", np.nan)),
                            "surrogate_subsample_n": float(eig_info.get("surrogate_subsample_n", np.nan)),
                            "surrogate_refine_iter": float(eig_info.get("surrogate_refine_iter", np.nan)),
                            "surrogate_n_iter": float(eig_info.get("surrogate_n_iter", np.nan)),
                            "surrogate_time_eigenspace": float(eig_info.get("surrogate_time_eigenspace", np.nan)),
                            "surrogate_time_refine": float(eig_info.get("surrogate_time_refine", np.nan)),
                            "surrogate_precompute": float(eig_info.get("surrogate_precompute", np.nan)),
                            "status": "error",
                            "error": tb,
                        }
                    )

full_df = pd.DataFrame(full_rows)
print("Full rows:", len(full_df))
full_df

Full rows: 5


,mode,eps,top_q,eig_method,m,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,...,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
0,gpu_precond,0.000001,64,subspace_iter,181,0.002602,NaN,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,gpu_precond,0.000001,64,sur_combo_g0p03_e100p0_s0p3,181,0.002602,4.297697e-12,0.000001,0.0,0.0,...,0.0001,5.0,30000.0,1.0,1.0,0.231696,5.988136,0.010486,ok,
2,gpu_precond,0.000001,64,sur_combo_g0p04_e100p0_s0p3,181,0.002601,4.474187e-11,0.000004,0.0,0.0,...,0.0001,7.0,30000.0,1.0,1.0,0.229012,6.895493,0.013045,ok,
3,gpu_precond,0.000001,64,sur_combo_g0p1_e100p0_s0p3,181,0.002602,1.047270e-11,0.000002,0.0,0.0,...,0.0001,18.0,30000.0,1.0,1.0,0.248154,6.979881,0.011803,ok,
4,gpu_precond,0.000001,64,custom_rand_lanczos,181,0.002601,2.571746e-11,0.000003,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,


In [36]:
cols = [
    "mode",
    "eig_method",
    "eps",
    "top_q",
    "m",
    "nufft_stage",
    "device_name",
    "surrogate_tag",
    "surrogate_eps",
    "surrogate_m",
    "surrogate_subsample_n",
    "surrogate_refine_iter",
    "surrogate_n_iter",
    "surrogate_time_eigenspace",
    "surrogate_time_refine",
    "surrogate_precompute",
    "status",
    "rmse_test",
    "beta_mse",
    "beta_mae",
    "yhat_mse",
    "yhat_mae",
    "cg_iters",
    "cg_relres",
    "n_matvec",
    "t_matvec_avg",
    "t_matvec_total",
    "n_precond",
    "t_precond_total",
    "t_precond_avg",
    "time_precompute",
    "time_eigenspace",
    "time_precond_build",
    "eig_residual_fro",
    "eig_residual_fro_rel",
    "time_solve",
    "t_cg_iter_avg",
    "time_predict",
    "wall_s_total",
    "error",
]
for col in cols:
    if col not in full_df.columns:
        full_df[col] = np.nan

with pd.option_context("display.max_columns", None, "display.width", 200):
    display(full_df[cols].sort_values(["eps", "mode", "top_q", "eig_method"]))

,mode,eig_method,eps,top_q,m,nufft_stage,device_name,surrogate_tag,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,cg_iters,cg_relres,n_matvec,t_matvec_avg,t_matvec_total,n_precond,t_precond_total,t_precond_avg,time_precompute,time_eigenspace,time_precond_build,eig_residual_fro,eig_residual_fro_rel,time_solve,t_cg_iter_avg,time_predict,wall_s_total,error
4,gpu_precond,custom_rand_lanczos,0.000001,64,181,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,0.002601,2.571746e-11,0.000003,0.0,0.0,131,7.116335e-07,132,0.013957,1.842359,131,10.694047,0.081634,0.174621,20.853558,0.000531,16.426273,0.001138,12.772638,0.097501,0.014214,33.815563,
0,gpu_precond,subspace_iter,0.000001,64,181,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,0.002602,NaN,NaN,0.0,0.0,133,9.223886e-07,134,0.013893,1.861696,133,5.693920,0.042811,0.174621,12.915861,0.000763,103.033257,0.007141,7.781782,0.058510,0.050245,20.923272,
1,gpu_precond,sur_combo_g0p03_e100p0_s0p3,0.000001,64,181,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p03_e100p0_s0p3,0.0001,5.0,30000.0,1.0,1.0,0.231696,5.988136,0.010486,ok,0.002602,4.297697e-12,0.000001,0.0,0.0,147,9.157015e-07,148,0.014011,2.073690,147,3.958022,0.026925,0.174621,6.219832,0.001095,416.942968,0.028926,6.322221,0.043008,0.016292,12.734060,
2,gpu_precond,sur_combo_g0p04_e100p0_s0p3,0.000001,64,181,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p04_e100p0_s0p3,0.0001,7.0,30000.0,1.0,1.0,0.229012,6.895493,0.013045,ok,0.002601,4.474187e-11,0.000004,0.0,0.0,142,7.462662e-07,143,0.013862,1.982211,142,6.066125,0.042719,0.174621,7.124506,0.000463,205.573208,0.014253,8.268382,0.058228,0.016586,15.584557,
3,gpu_precond,sur_combo_g0p1_e100p0_s0p3,0.000001,64,181,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,combo_g0p1_e100p0_s0p3,0.0001,18.0,30000.0,1.0,1.0,0.248154,6.979881,0.011803,ok,0.002602,1.047270e-11,0.000002,0.0,0.0,140,8.203509e-07,141,0.014004,1.974516,140,11.296078,0.080686,0.174621,7.228035,0.000421,239.987123,0.016637,13.530825,0.096649,0.019253,20.953155,
